In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

state_dict = torch.load("/content/medical_model.pt", map_location=device)

print(type(state_dict))   # should be dict

<class 'collections.OrderedDict'>


In [ ]:
def build_input(row):
    age = str(row.get('age', ''))
    gender = str(row.get('gender', ''))

    return (
        f"Age: {age}. Gender: {gender}. " +   #  structured context
        str(row.get('title', '')) + " " +
        str(row.get('keywords', '')) * 2 + " " +
        str(row.get('abstract', '')) + " " +
        str(row.get('image_captions', ''))
    )

In [ ]:
df['input_text'] = df.apply(build_input, axis=1)

In [ ]:
df = pd.read_csv("/content/final_labeled_dataset.csv")

# basic cleaning
df = df.dropna(subset=['input_text', 'system_label', 'type_label'])
df = df.reset_index(drop=True)

print("Data shape:", df.shape)
df.head()

Data shape: (61437, 4)


,case_id,input_text,system_label,type_label
0,PMC3738355_01,Littoral cell angioma mimicking hepatic tumor ...,hepatobiliary,tumor
1,PMC5015624_01,Central retinal artery occlusion following las...,cardiovascular,vascular
2,PMC6381877_01,Promising Combination Therapy with Bevacizumab...,respiratory,tumor
3,PMC5287946_01,Malignant adenohypophysis spindle cell oncocyt...,multisystemic,tumor
4,PMC9106225_01,"A novel mucocele: Myxoglobulosis [['mucocele',...",ent,trauma


In [ ]:
SYSTEM_LABELS = [
    "neurological","cardiovascular","respiratory","musculoskeletal",
    "hematological","hepatobiliary","gastrointestinal","genitourinary",
    "renal","lymphatic","immunological","dermatological",
    "ophthalmic","ent","endocrine","multisystemic", "genetic"
]

TYPE_LABELS = [
    "tumor","infection","vascular","metabolic",
    "degenerative","trauma","autoimmune"
]

In [ ]:
system_labels = sorted(df['system_strong'].unique())
type_labels = sorted(df['type_strong'].unique())

print(system_labels)
print(type_labels)

In [ ]:
le_system = LabelEncoder()
le_type = LabelEncoder()

le_system.fit(SYSTEM_LABELS)
le_type.fit(TYPE_LABELS)

print("System classes:", le_system.classes_)
print("Type classes:", le_type.classes_)

System classes: ['cardiovascular' 'dermatological' 'endocrine' 'ent' 'gastrointestinal'
 'genetic' 'genitourinary' 'hematological' 'hepatobiliary' 'immunological'
 'lymphatic' 'multisystemic' 'musculoskeletal' 'neurological' 'ophthalmic'
 'renal' 'respiratory']
Type classes: ['autoimmune' 'degenerative' 'infection' 'metabolic' 'trauma' 'tumor'
 'vascular']


In [ ]:
print("Unique system labels in data:", df['system_label'].unique())
print("Unique type labels in data:", df['type_label'].unique())

Unique system labels in data: ['hepatobiliary' 'cardiovascular' 'respiratory' 'multisystemic' 'ent'
 'neurological' 'musculoskeletal' 'immunological' 'ophthalmic' 'genetic'
 'gastrointestinal' 'dermatological' 'hematological' 'renal' 'lymphatic'
 'genitourinary' 'endocrine']
Unique type labels in data: ['tumor' 'vascular' 'trauma' 'infection' 'degenerative' 'autoimmune'
 'metabolic']


In [ ]:
class MedicalDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['input_text'].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=256,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze()
        }

In [ ]:
full_dataset = MedicalDataset(df, tokenizer)
full_loader = DataLoader(full_dataset, batch_size=16)

In [ ]:
class MultiTaskModel(nn.Module):
    def __init__(self, num_system, num_type):
        super().__init__()
        self.encoder = AutoModel.from_pretrained("distilbert-base-uncased")
        hidden = self.encoder.config.hidden_size

        self.system_head = nn.Linear(hidden, num_system)
        self.type_head = nn.Linear(hidden, num_type)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0]
        return self.system_head(cls), self.type_head(cls)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiTaskModel(
    num_system=len(le_system.classes_),
    num_type=len(le_type.classes_)
).to(device)

model.load_state_dict(torch.load("medical_model.pt", map_location=device))
model.eval()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MultiTaskModel(
  (encoder): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Line

In [ ]:
sys_preds, type_preds = [], []
sys_confs, type_confs = [], []

with torch.no_grad():
    for batch in tqdm(full_loader):

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        sys_logits, type_logits = model(input_ids, attention_mask)

        sys_probs = F.softmax(sys_logits, dim=1)
        type_probs = F.softmax(type_logits, dim=1)

        sys_pred = torch.argmax(sys_probs, dim=1).cpu().numpy()
        type_pred = torch.argmax(type_probs, dim=1).cpu().numpy()

        sys_conf = torch.max(sys_probs, dim=1).values.cpu().numpy()
        type_conf = torch.max(type_probs, dim=1).values.cpu().numpy()

        sys_preds.extend(sys_pred)
        type_preds.extend(type_pred)

        sys_confs.extend(sys_conf)
        type_confs.extend(type_conf)

100%|██████████| 3840/3840 [08:49<00:00,  7.25it/s]


In [ ]:
df['system_pred'] = le_system.inverse_transform(sys_preds)
df['type_pred'] = le_type.inverse_transform(type_preds)

df['system_conf'] = sys_confs
df['type_conf'] = type_confs

In [ ]:
system_changes = (df['system_label'] != df['system_pred']).sum()
type_changes = (df['type_label'] != df['type_pred']).sum()

total = len(df)

print("Total samples:", total)
print("System changes:", system_changes)
print("Type changes:", type_changes)

print("System change %:", system_changes / total * 100)
print("Type change %:", type_changes / total * 100)

Total samples: 61437
System changes: 6869
Type changes: 4744
System change %: 11.180558946563146
Type change %: 7.721731204323127


In [ ]:
df_diff = df[
    (df['system_label'] != df['system_pred']) |
    (df['type_label'] != df['type_pred'])
]

for _, row in df_diff.sample(10).iterrows():
    print("="*100)
    print("TEXT:", row['input_text'][:300])
    print("OLD SYSTEM:", row['system_label'], "→ NEW:", row['system_pred'])
    print("OLD TYPE:", row['type_label'], "→ NEW:", row['type_pred'])

TEXT: Case report: Equine metacarpophalangeal joint partial and full thickness defects treated with allogenic equine synovial membrane mesenchymal stem/stromal cell combined with umbilical cord mesenchymal stem/stromal cell conditioned medium [['case report', 'cell-based medicinal product', 'equine', 'ost
OLD SYSTEM: multisystemic → NEW: musculoskeletal
OLD TYPE: tumor → NEW: tumor
TEXT: Epiphrenic Diverticulum in an Infant with Congenital Esophageal Stenosis Associated with Esophageal Atresia [['esophageal atresia', 'children', 'congenital esophageal stenosis', 'epiphrenic diverticula', 'esophageal diverticula']][['esophageal atresia', 'children', 'congenital esophageal stenosis', 
OLD SYSTEM: multisystemic → NEW: gastrointestinal
OLD TYPE: degenerative → NEW: degenerative
TEXT: Severe G6PD deficiency leads to recurrent infections and defects in ROS production: Case report and literature review [['g6pd gene', 'nf-κb pathway', 'ros', 'chromosome inactivation', 'infection']][['g6pd gene

In [ ]:
mask_sys = (df['system_pred'] != df['system_label']) & (df['system_conf'] > 0.85)
mask_type = (df['type_pred'] != df['type_label']) & (df['type_conf'] > 0.80)

print("High-confidence system fixes:", mask_sys.sum())
print("High-confidence type fixes:", mask_type.sum())

High-confidence system fixes: 4320
High-confidence type fixes: 1972


In [ ]:
df['system_strong'] = df['system_label']
df['type_strong'] = df['type_label']

df.loc[mask_sys, 'system_strong'] = df.loc[mask_sys, 'system_pred']
df.loc[mask_type, 'type_strong'] = df.loc[mask_type, 'type_pred']

In [ ]:
df.to_csv("/content/strong_labeled_dataset.csv", index=False)

In [ ]:
import random

def inspect_random_samples(df, n=10):
    samples = df.sample(n)

    for i, row in samples.iterrows():
        print("="*100)
        print("TEXT:\n", row['input_text'][:500])

        print("\n--- LABELS ---")
        print("OLD SYSTEM :", row['system_label'])
        print("PRED SYSTEM:", row['system_pred'], "| CONF:", round(row['system_conf'], 3))
        print("FINAL SYSTEM:", row['system_strong'])

        print("\nOLD TYPE :", row['type_label'])
        print("PRED TYPE:", row['type_pred'], "| CONF:", round(row['type_conf'], 3))
        print("FINAL TYPE:", row['type_strong'])

In [ ]:
inspect_random_samples(df, n=50)

TEXT:
 Bilateral Vertebral Artery Dissection: A Case Report with Literature Review nannan Vertebral artery dissection (VAD) is a rare cause of ischemic stroke in young patients. The largely nonspecific symptoms and delayed presentation pose a serious diagnostic challenge. Medical management with either anticoagulation or antiplatelet therapy is recommended, but there are no reports of successful dual therapy. We report a case of spontaneous bilateral vertebral artery dissections (VADs) treated with bot

--- LABELS ---
OLD SYSTEM : neurological
PRED SYSTEM: neurological | CONF: 0.999
FINAL SYSTEM: neurological

OLD TYPE : vascular
PRED TYPE: vascular | CONF: 0.994
FINAL TYPE: vascular
TEXT:
 TKI-induced pure red cell aplasia: first case report of pure red cell aplasia with both imatinib and nilotinib [['chronic myeloid leukemia', 'gastrointestinal stromal tumors', 'imatinib', 'pure red cell aplasia', 'tyrosine kinase inhibitors']][['chronic myeloid leukemia', 'gastrointestinal stromal t

In [ ]:
df = pd.read_csv("/content/strong_labeled_dataset.csv")



print("Data shape:", df.shape)
df.head()

Data shape: (61437, 10)


,case_id,input_text,system_label,type_label,system_pred,type_pred,system_conf,type_conf,system_strong,type_strong
0,PMC3738355_01,Littoral cell angioma mimicking hepatic tumor ...,hepatobiliary,tumor,hepatobiliary,tumor,0.997788,0.997832,hepatobiliary,tumor
1,PMC5015624_01,Central retinal artery occlusion following las...,cardiovascular,vascular,cardiovascular,vascular,0.997213,0.995973,cardiovascular,vascular
2,PMC6381877_01,Promising Combination Therapy with Bevacizumab...,respiratory,tumor,respiratory,tumor,0.999103,0.998234,respiratory,tumor
3,PMC5287946_01,Malignant adenohypophysis spindle cell oncocyt...,multisystemic,tumor,multisystemic,tumor,0.990248,0.999513,multisystemic,tumor
4,PMC9106225_01,"A novel mucocele: Myxoglobulosis [['mucocele',...",ent,trauma,ent,trauma,0.996896,0.972045,ent,trauma


In [ ]:
import zipfile
import os

zip_root = "/content/drive/MyDrive/meddiag"
extract_root = "meddiag_extracted"

os.makedirs(extract_root, exist_ok=True)

for file in os.listdir(zip_root):
    if file.endswith(".zip"):
        zip_path = os.path.join(zip_root, file)

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_root)

        print(f"Extracted: {file}")

Extracted: PMC3.zip
Extracted: PMC2.zip
Extracted: PMC1.zip
Extracted: PMC6.zip
Extracted: PMC8.zip
Extracted: PMC5.zip
Extracted: PMC4.zip
Extracted: PMC7.zip
Extracted: PMC9.zip


In [ ]:
import os

root = "meddiag_extracted"

for folder in os.listdir(root):
    folder_path = os.path.join(root, folder)

    if not os.path.isdir(folder_path):
        continue

    print("Folder:", folder)
    print(os.listdir(folder_path)[:5])
    break

Folder: PMC1
['PMC15', 'PMC11', 'PMC10', 'PMC13', 'PMC16']


In [ ]:
import os
from collections import defaultdict

image_dict = defaultdict(list)

root = "meddiag_extracted"

for dirpath, dirnames, filenames in os.walk(root):

    for img in filenames:
        if img.endswith((".jpg", ".png", ".jpeg", ".webp")):

            case_id = img.split("_")[0]  # PMC1234567

            full_path = os.path.join(dirpath, img)
            image_dict[case_id].append(full_path)

In [ ]:
print("Total image cases:", len(image_dict))
print("Sample keys:", list(image_dict.keys())[:10])

Total image cases: 30526
Sample keys: ['PMC1550567', 'PMC1501118', 'PMC1502069', 'PMC1502064', 'PMC11722611', 'PMC11227686', 'PMC11222310', 'PMC11002164', 'PMC11091343', 'PMC11749671']


In [ ]:
df['case_id_clean'] = df['case_id'].apply(lambda x: x.split('_')[0])

df['image_paths'] = df['case_id_clean'].map(image_dict)

df['image_paths'] = df['image_paths'].apply(
    lambda x: x if isinstance(x, list) else []
)

In [ ]:
total_cases = len(df)
cases_with_images = (df['image_paths'].apply(len) > 0).sum()

print("Total cases:", total_cases)
print("Cases with images:", cases_with_images)
print("Coverage %:", cases_with_images / total_cases * 100)

Total cases: 61437
Cases with images: 26184
Coverage %: 42.61926851897066


In [ ]:
df[df['image_paths'].apply(len) > 0][['case_id','image_paths']].head(5)

,case_id,image_paths
1,PMC5015624_01,[meddiag_extracted/PMC5/PMC50/PMC5015624_OC-05...
2,PMC6381877_01,[meddiag_extracted/PMC6/PMC63/PMC6381877_cro-0...
3,PMC5287946_01,[meddiag_extracted/PMC5/PMC52/PMC5287946_medi-...
4,PMC9106225_01,[meddiag_extracted/PMC9/PMC91/PMC9106225_JOMFP...
5,PMC7276389_01,[meddiag_extracted/PMC7/PMC72/PMC7276389_gr2_a...


In [ ]:
df['num_images'] = df['image_paths'].apply(len)

print(df['num_images'].describe())

count    61437.000000
mean         1.783795
std          3.026386
min          0.000000
25%          0.000000
50%          0.000000
75%          3.000000
max         47.000000
Name: num_images, dtype: float64


In [ ]:
from torchvision import transforms

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
from PIL import Image
import torch

class MultiModalDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # TEXT
        text = str(row['input_text'])

        enc = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        input_ids = enc['input_ids'].squeeze()
        attention_mask = enc['attention_mask'].squeeze()

        # IMAGE
        image_paths = row['image_paths']

        if len(image_paths) > 0:
            try:
                img = Image.open(image_paths[0]).convert("RGB")
                img = image_transform(img)
            except:
                img = torch.zeros(3, 224, 224)
        else:
            img = torch.zeros(3, 224, 224)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'image': img,
            'system_label': row['system_label'],
            'type_label': row['type_label']
        }

In [ ]:
class MultiModalDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # TEXT
        text = str(row['input_text'])

        enc = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        input_ids = enc['input_ids'].squeeze()
        attention_mask = enc['attention_mask'].squeeze()

        # IMAGE
        image_paths = row['image_paths']

        if len(image_paths) > 0:
            try:
                img = Image.open(image_paths[0]).convert("RGB")
                img = image_transform(img)
            except:
                img = torch.zeros(3, 224, 224)
        else:
            img = torch.zeros(3, 224, 224)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'image': img,
            'system_label': torch.tensor(row['system_label_enc']),
            'type_label': torch.tensor(row['type_label_enc'])
        }

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_system = LabelEncoder()
le_type = LabelEncoder()

df['system_label_enc'] = le_system.fit_transform(df['system_strong'])
df['type_label_enc'] = le_type.fit_transform(df['type_strong'])

num_system = len(le_system.classes_)
num_type = len(le_type.classes_)

print("System classes:", le_system.classes_)
print("Type classes:", le_type.classes_)

System classes: ['cardiovascular' 'dermatological' 'endocrine' 'ent' 'gastrointestinal'
 'genetic' 'genitourinary' 'hematological' 'hepatobiliary' 'immunological'
 'lymphatic' 'multisystemic' 'musculoskeletal' 'neurological' 'ophthalmic'
 'renal' 'respiratory']
Type classes: ['autoimmune' 'degenerative' 'infection' 'metabolic' 'trauma' 'tumor'
 'vascular']


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torchvision import models, transforms
from PIL import Image

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from tqdm import tqdm

In [ ]:
df = pd.read_csv("/content/strong_labeled_dataset.csv")

df = df.dropna(subset=['input_text', 'system_strong', 'type_strong']).reset_index(drop=True)

print(df.shape)

(61437, 10)


In [ ]:
le_system = LabelEncoder()
le_type = LabelEncoder()

df['system_label_enc'] = le_system.fit_transform(df['system_strong'])
df['type_label_enc'] = le_type.fit_transform(df['type_strong'])

num_system = len(le_system.classes_)
num_type = len(le_type.classes_)

print(num_system, num_type)

17 7


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
class MultiModalDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # TEXT
        text = str(row['input_text'])

        enc = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        input_ids = enc['input_ids'].squeeze()
        attention_mask = enc['attention_mask'].squeeze()

        # IMAGE
        image_paths = row['image_paths']

        if isinstance(image_paths, list) and len(image_paths) > 0:
            try:
                img = Image.open(image_paths[0]).convert("RGB")
                img = image_transform(img)
            except:
                img = torch.zeros(3, 224, 224)
        else:
            img = torch.zeros(3, 224, 224)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'image': img,
            'system_label': torch.tensor(row['system_label_enc']),
            'type_label': torch.tensor(row['type_label_enc'])
        }

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

train_dataset = MultiModalDataset(train_df, tokenizer)
val_dataset = MultiModalDataset(val_df, tokenizer)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, num_workers=4, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
class MultiModalModel(nn.Module):
    def __init__(self, num_system, num_type):
        super().__init__()

        # TEXT encoder
        self.text_encoder = AutoModel.from_pretrained("distilbert-base-uncased")
        text_dim = self.text_encoder.config.hidden_size

        # IMAGE encoder
        resnet = models.resnet18(pretrained=True)
        self.image_encoder = nn.Sequential(*list(resnet.children())[:-1])

        # 🔥 FREEZE IMAGE ENCODER
        for param in self.image_encoder.parameters():
            param.requires_grad = False

        img_dim = 512

        # FUSION
        self.fc = nn.Linear(text_dim + img_dim, 256)

        # HEADS
        self.system_head = nn.Linear(256, num_system)
        self.type_head = nn.Linear(256, num_type)

    def forward(self, input_ids, attention_mask, image):
        # TEXT
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_emb = text_out.last_hidden_state[:, 0]

        # IMAGE
        img_feat = self.image_encoder(image)
        img_feat = img_feat.view(img_feat.size(0), -1)

        # FUSION
        combined = torch.cat([text_emb, img_feat], dim=1)
        x = self.fc(combined)

        return self.system_head(x), self.type_head(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiModalModel(num_system, num_type).to(device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You ca

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 121MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

In [ ]:
from collections import defaultdict
import os

image_dict = defaultdict(list)

root = "meddiag_extracted"

for dirpath, _, filenames in os.walk(root):
    for img in filenames:
        if img.endswith((".jpg", ".png", ".jpeg", ".webp")):
            case_id = img.split("_")[0]
            full_path = os.path.join(dirpath, img)
            image_dict[case_id].append(full_path)

In [ ]:
df['case_id_clean'] = df['case_id'].apply(lambda x: x.split('_')[0])

df['image_paths'] = df['case_id_clean'].map(image_dict)

df['image_paths'] = df['image_paths'].apply(
    lambda x: x if isinstance(x, list) else []
)

In [ ]:
print("Column exists:", 'image_paths' in df.columns)

print("Samples with images:",
      (df['image_paths'].apply(len) > 0).sum())

Column exists: True
Samples with images: 26184


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

In [ ]:
train_dataset = MultiModalDataset(train_df, tokenizer)
val_dataset = MultiModalDataset(val_df, tokenizer)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,   # 🔥 change from 4 → 2
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    num_workers=2,
    pin_memory=True
)

In [ ]:
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        images = batch['image'].to(device)

        sys_labels = batch['system_label'].to(device)
        type_labels = batch['type_label'].to(device)

        optimizer.zero_grad()

        sys_logits, type_logits = model(input_ids, attention_mask, images)

        loss_sys = criterion(sys_logits, sys_labels)
        loss_type = criterion(type_logits, type_labels)

        loss = loss_sys + loss_type

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        progress_bar.set_postfix(loss=loss.item())

    print(f"\nEpoch {epoch+1} DONE | Total Loss: {total_loss:.4f}")

Epoch 1: 100%|██████████| 6912/6912 [24:26<00:00,  4.71it/s, loss=0.911]



Epoch 1 DONE | Total Loss: 8126.3706


Epoch 2: 100%|██████████| 6912/6912 [24:32<00:00,  4.69it/s, loss=0.0568]



Epoch 2 DONE | Total Loss: 3979.9135


Epoch 3: 100%|██████████| 6912/6912 [24:30<00:00,  4.70it/s, loss=0.164]


Epoch 3 DONE | Total Loss: 2689.4233


In [ ]:
model.eval()

sys_preds, sys_true = [], []
type_preds, type_true = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        images = batch['image'].to(device)

        sys_labels = batch['system_label'].to(device)
        type_labels = batch['type_label'].to(device)

        sys_logits, type_logits = model(input_ids, attention_mask, images)

        sys_pred = torch.argmax(sys_logits, dim=1)
        type_pred = torch.argmax(type_logits, dim=1)

        sys_preds.extend(sys_pred.cpu().numpy())
        sys_true.extend(sys_labels.cpu().numpy())

        type_preds.extend(type_pred.cpu().numpy())
        type_true.extend(type_labels.cpu().numpy())

print("System F1:", f1_score(sys_true, sys_preds, average='macro'))
print("Type F1:", f1_score(type_true, type_preds, average='macro'))

System F1: 0.820222103514724
Type F1: 0.860137979274082


In [ ]:
torch.save(model.state_dict(), "multimodal_model.pt")

In [ ]:
from google.colab import files

files.download("multimodal_model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>